# Files — Claude Managed Agents (CLI)

The same flow as [`04-files/`](../04-files/), driven with the **`ant`** CLI. Files move **into** the sandbox and the agent's work comes back **out**:

```
ant files upload --file x.md              →  file_id, stored in your organization's Files storage
ant beta:sessions create --resource ...   →  a read-only copy mounted at /mnt/session/uploads/...
ant beta:sessions:resources add           →  mount another file into a running session
agent writes /mnt/session/outputs/        →  each output file shows up in the Files API, scoped to the session
ant beta:files list --scope-id ...        →  find the outputs;  ant files download  →  save them locally
```

The task: review the supplier contracts in [`sample-data/contracts/`](../sample-data/contracts/) and produce a summary spreadsheet and a risk report.

> Prerequisite: the `ant` CLI (>= 1.40) installed and authenticated — see [`01-basics-cli/`](../01-basics-cli/).

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# `ant` is a separate process from this notebook, so a bit of setup:
#  - it doesn't read the root .env — load the API key into os.environ, which
#    every `!ant` call below inherits;
#  - Jupyter sets FORCE_COLOR=1, which makes ant wrap its JSON output in
#    ANSI color codes and break parsing — clear it so the output stays plain;
#  - `!` runs commands in a pseudo-terminal, so ant probes it with escape
#    sequences that leak into the cell output — TERM=dumb turns that off.
load_dotenv(find_dotenv())
os.environ.pop("FORCE_COLOR", None)
os.environ.pop("CLICOLOR_FORCE", None)
os.environ["NO_COLOR"] = "1"
os.environ["TERM"] = "dumb"
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found — add it to the root .env or run `ant auth login`."
)
print("Credential loaded, color disabled — ant is ready")

## 2. Environment and Agent

The system prompt tells the agent **where things are**: inputs under `/mnt/session/uploads/`, deliverables under `/mnt/session/outputs/` — the only directory whose files come back through the API.

In [ ]:
%%writefile environment.yaml
# yaml-language-server: $schema=https://platform.claude.com/schemas/ant/beta/environment.json
name: files-env-cli
config:
  type: cloud
  networking:
    type: unrestricted

In [ ]:
%%writefile agent.md
---
name: Contract Analyst (CLI)
model: claude-opus-5-5
tools:
  - type: agent_toolset_20260401
    configs:
      - name: web_search
        enabled: false
      - name: web_fetch
        enabled: false
---

You are a contract analyst for Brightloom Inc.
Contracts to review are mounted read-only under /mnt/session/uploads/contracts/.
Write every deliverable to /mnt/session/outputs/ — files anywhere else are not returned to the user.
Quote clause numbers when you cite a contract. Keep chat replies short; put the detail in the files.

In [ ]:
import json

!ant apply environment.yaml agent.md --yes --lock-file claude-lock.json

lock = json.load(open("claude-lock.json"))["resources"]
env_id = lock["./environment.yaml"]["id"]
agent_id, agent_version = lock["./agent.md"]["id"], int(lock["./agent.md"]["version"])
print(f"Environment : {env_id}")
print(f"Agent       : {agent_id}  (v{agent_version})")

## 3. Upload Files

`ant files upload` sends a file to the **Files API** and returns its `file_id`. Upload once, then mount the same file into any number of sessions. The filename travels with the file, so give it a **descriptive name** — the agent sees it.

We upload five of the six contracts now and keep the last one for later.

In [ ]:
from pathlib import Path

contract_paths = sorted(Path("../sample-data/contracts").glob("*.md"))
initial, late = contract_paths[:5], contract_paths[5]

uploaded = {}  # filename -> file_id
for path in initial:
    _id = !ant files upload --file {path} --transform id -r  # type: ignore
    uploaded[path.name] = _id[0]
    print(f"{_id[0]}  {path.name}")

## 4. Mount Files into a Session

Each file becomes a `--resource` flag on `ant beta:sessions create` — repeat the flag once per file:

- `mount_path` is rooted under `/mnt/session/uploads/`, so `/contracts/x.md` lands at `/mnt/session/uploads/contracts/x.md`. Without it, the file lands at `/mnt/session/uploads/<file_id>`.
- Mounted files are **read-only copies**; up to 500 per session.

In [ ]:
agent_ref = json.dumps({"type": "agent", "id": agent_id, "version": agent_version})
resource_flags = " ".join(
    "--resource '" + json.dumps({"type": "file", "file_id": file_id, "mount_path": f"/contracts/{name}"}) + "'"
    for name, file_id in uploaded.items()
)

_session = !ant beta:sessions create --agent '{agent_ref}' --environment-id {env_id} --title "Contract review CLI" {resource_flags} --transform id -r  # type: ignore
session_id = _session[0]
print("Session ID:", session_id)

_resources = !ant beta:sessions:resources list --session-id {session_id} --transform '{{id,mount_path}}' --format jsonl  # type: ignore
for line in _resources:
    resource = json.loads(line)
    print(f"  {resource['id']}  ->  {resource['mount_path']}")

## 5. Run the Review

The streaming loop from module 03, trimmed: no custom tools and an `always_allow` toolset, so nothing waits on us. It prints each tool call so you can watch the agent work through the files.

In [ ]:
import subprocess


def send(session_id: str, *events: dict) -> None:
    subprocess.run(
        ["ant", "beta:sessions:events", "send", "--session-id", session_id],
        input=json.dumps({"events": list(events)}),
        stdout=subprocess.DEVNULL, text=True, check=True,
    )


def events(stream):
    """Yield each event from `ant ... stream --format json` as a dict."""
    buf = []
    for line in stream.stdout:
        buf.append(line)
        if line.rstrip("\n") == "}":
            yield json.loads("".join(buf))
            buf = []


def run_turn(session_id: str, text: str) -> None:
    """Send one user message and stream until the turn ends."""
    stream = subprocess.Popen(
        ["ant", "beta:sessions:events", "stream", "--session-id", session_id, "--format", "json"],
        stdout=subprocess.PIPE, text=True, bufsize=1,
    )
    send(session_id, {"type": "user.message", "content": [{"type": "text", "text": text}]})
    for event in events(stream):
        match event["type"]:
            case "agent.message":
                for block in event["content"]:
                    if block["type"] == "text":
                        print(f"Agent: {block['text']}")
            case "agent.tool_use":
                i = event["input"]
                detail = i.get("command") or i.get("file_path") or i.get("pattern") or ""
                print(f"[{event['name']}] {str(detail)[:100]}")
            case "session.status_idle":
                if event["stop_reason"]["type"] == "requires_action":
                    continue
                print(f"[idle: {event['stop_reason']['type']}]")
                break
            case "session.status_terminated":
                print("[terminated]")
                break
    stream.terminate()

In [ ]:
run_turn(
    session_id,
    "Review every contract under /mnt/session/uploads/contracts/. Write two files to /mnt/session/outputs/: "
    "summary.csv with columns supplier,effective_date,auto_renews,non_renewal_notice_days,"
    "termination_for_convenience,liability_cap,change_of_control,governing_law — one row per contract; "
    "and report.md, a one-page risk report that ranks the contracts from most to least risky for Brightloom "
    "and explains the top risks of each.",
)

## 6. Add a File to a Running Session

`ant beta:sessions:resources add` mounts another file into a session that is already running (`... resources delete --resource-id` removes one). The sandbox persists between turns, so the agent updates its earlier files instead of starting over.

In [ ]:
_id = !ant files upload --file {late} --transform id -r  # type: ignore
uploaded[late.name] = _id[0]

!ant beta:sessions:resources add --session-id {session_id} --type file --file-id {_id[0]} --mount-path /contracts/{late.name} --transform mount_path -r

In [ ]:
run_turn(
    session_id,
    f"A new contract was just added: /mnt/session/uploads/contracts/{late.name}. "
    "Review it and update both summary.csv and report.md in /mnt/session/outputs/ to include it.",
)

## 7. Download the Outputs

`ant beta:files list --scope-id <session>` lists the files scoped to the session. Scope filtering needs the Managed Agents beta header, so pass `--beta managed-agents-2026-04-01`.

The list includes the **session copies of the files you mounted** — those report `downloadable: false` — and the agent's outputs, which are downloadable. Outputs can appear a few seconds after the idle, so we poll briefly.

In [ ]:
import time

EXPECTED = {"summary.csv", "report.md"}


def session_outputs(session_id: str, expected: set[str], timeout: float = 60) -> dict:
    """Return {filename: file_id} for the session's downloadable files, waiting until `expected` are listed."""
    deadline = time.monotonic() + timeout
    while True:
        listing = subprocess.run(
            ["ant", "beta:files", "list", "--scope-id", session_id, "--beta", "managed-agents-2026-04-01",
             "--transform", "{id,filename,downloadable,created_at}", "--format", "jsonl"],
            capture_output=True, text=True, check=True,
        ).stdout
        files = [json.loads(line) for line in listing.splitlines() if line.strip()]
        newest = {}
        for f in sorted((f for f in files if f["downloadable"]), key=lambda f: f["created_at"]):
            newest[f["filename"]] = f["id"]   # later entries win
        if expected <= newest.keys() or time.monotonic() > deadline:
            return newest
        time.sleep(3)


outputs = session_outputs(session_id, EXPECTED)
Path("outputs").mkdir(exist_ok=True)
for name, file_id in outputs.items():
    !ant files download --file-id {file_id} --output outputs/{name}

In [ ]:
from IPython.display import Markdown, display

print(Path("outputs/summary.csv").read_text())
display(Markdown(Path("outputs/report.md").read_text()))

## 8. Clean Up

Uploaded files stay in your organization's storage until you delete them; the session keeps its own copies, so deleting the uploads doesn't break it.

In [ ]:
for name, file_id in uploaded.items():
    !ant files delete --file-id {file_id} --transform id -r
    print(f"  deleted {name}")

## Summary

```
ant files upload --file x.md --transform id -r          →  file_id
ant beta:sessions create ... --resource '{type: file,   →  read-only at /mnt/session/uploads/<mount_path>
    file_id: ..., mount_path: ...}'   (repeat per file)
ant beta:sessions:resources add / list / delete          →  change a running session's files
agent writes /mnt/session/outputs/...                    →  Files API, scoped to the session
ant beta:files list --scope-id <session> --beta managed-agents-2026-04-01
ant files download --file-id <id> --output <path>
```

### CLI vs SDK, same lesson

| | CLI (this module) | SDK ([`04-files/`](../04-files/)) |
|---|---|---|
| Upload | `ant files upload --file` | `client.files.upload(file=path)` |
| Mount at creation | `--resource '{...}'` per file | `resources=[...]` |
| Mount later | `ant beta:sessions:resources add` | `client.beta.sessions.resources.add()` |
| List outputs | `ant beta:files list --scope-id ... --beta ...` | `client.beta.files.list(scope_id=..., betas=[...])` |
| Download | `ant files download --output` | `client.files.download(id).write_to_file()` |

### Next steps
- **05-session-control** — steer, interrupt, reconnect, and cap spend with a session budget
- **06-subagents** — delegate to other agents in session threads
- **07-dynamic-workflows** — let the agent write a workflow that runs many agents in parallel